## 1. Subscribe to the model package

To subscribe to the model package:
1. Open the model package listing page [MedRouter]()
1. On the AWS Marketplace listing, click on the **Continue to subscribe** button.
1. On the **Subscribe to this software** page, review and click on **"Accept Offer"** if you and your organization agrees with EULA, pricing, and support terms.
1. Once you click on **Continue to configuration button** and then choose a **region**, you will see a **Product Arn** displayed. This is the model package ARN that you need to specify while creating a deployable model using Boto3. Copy the ARN corresponding to your region and specify the same in the following cell.

In [ ]:
model_package_arn = "<Customer to specify Model package ARN corresponding to their AWS region>"

In [ ]:
import json
import re
from sagemaker import ModelPackage
import sagemaker as sage
from sagemaker import get_execution_role
import boto3
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

sagemaker_session = sage.Session()
s3_bucket = sagemaker_session.default_bucket()
region = sagemaker_session.boto_region_name
account_id = boto3.client("sts").get_caller_identity().get("Account")
role = get_execution_role()

sagemaker = boto3.client("sagemaker")
s3_client = sagemaker_session.boto_session.client("s3")
sm_runtime = boto3.client("sagemaker-runtime")

In [3]:
model_name = "MedRouter"

real_time_inference_instance_type = 'ml.p4d.24xlarge'
batch_transform_inference_instance_type = "ml.g5.48xlarge"

## 2. Create a deployable model from the model package.

In [4]:
model = ModelPackage(
    role=role,
    model_package_arn=model_package_arn,
    sagemaker_session=sagemaker_session,
)

## Model Configuration Documentation

### Inference Parameters (Fixed)

Inference parameters are fixed server-side and cannot be overridden per request.

| Parameter | Value | Description |
|---|---|---|
| **`temperature`** | `1.0` | Sampling temperature |
| **`top_p`** | `0.95` | Nucleus sampling threshold |
| **`repetition_penalty`** | `1.05` | Discourages token repetition |
| **`max_tokens`** | `16000` | Maximum generated tokens per response |

---

### GPU Configuration

`gpu_memory_utilization` is automatically detected at startup based on per-GPU VRAM:

| Per-GPU VRAM | `gpu_memory_utilization` |
|---|---|
| ≥ 40 GB | `0.90` |
| < 40 GB | `0.75` |

`tensor_parallel_size` is automatically set to the number of available GPUs.

---

### Input Format

MEDRouter exposes a `/invocations` endpoint. The request body is a standard chat-completions payload:

```json
{
    "messages": [
        {"role": "system", "content": "You are an expert medical AI assistant."},
        {"role": "user",   "content": "<medical question>"}
    ],
    "stream": false
}
```

| Field | Type | Required | Description |
|---|---|---|---|
| `messages` | array | Yes | Conversation messages |
| `stream` | boolean | No | `false` for batch transform. `true` for real-time streaming. |
| `model` | string | No | Accepted but ignored. Inference parameters are fixed server-side. |

The `system` message is optional. If omitted, a default medical system prompt is applied.

Inference parameters (temperature, top_p, max_tokens, etc.) are fixed server-side and cannot be overridden per request.


## 3. Create a SageMaker Endpoint

If you want to understand how real-time inference with Amazon SageMaker works, see [Documentation](https://docs.aws.amazon.com/sagemaker/latest/dg/how-it-works-hosting.html).

In [ ]:
predictor = model.deploy(
    initial_instance_count=1,
    instance_type=real_time_inference_instance_type,
    endpoint_name=model_name,
    model_data_download_timeout=3600,
)

In [5]:
prompt1 = """\
A 95-year-old woman who is a resident at a long term care facility, got up from her chair, tripped on a rug, and fell on her right knee. She could not get up without assistance and complained of severe pain in her right hip and buttock. The nurse who evaluated her tried to stand her up, but when the patient tried to stand on her right leg, she dropped her left hip and lost her balance. The nurse then recognized that her patient had a foreshortened right leg fixed in the adducted position and a large swelling in her right buttock. At the receiving hospital, the patient was confused and, though she knew her name, she couldn’t remember the date and insists to leave the hospital immediately to see her family. Past medical history includes diabetes, congestive heart failure, and incontinence. She is currently taking metformin, lisinopril, hydrochlorothiazide, metoprolol, and oxybutynin. Physical exam confirmed the nurse’s findings. Radiographs proved the presence of a right posterior hip dislocation without fractures. What medication is most likely associated with this patient’s confusion?
A. Metformin
B. Oxybutynin
C. Metoprolol
D. Lisinopril"""

prompt2 = """\
A 47-year-old man comes to the physician because of severe retrosternal chest pain and shortness of breath for 45 minutes. He has dyslipidemia, hypertension, and type 2 diabetes mellitus. Current medications include hydrochlorothiazide, lisinopril, metformin, and atorvastatin. He has smoked 1 pack of cigarettes daily for 20 years. He appears pale and diaphoretic. His temperature is 37°C (98.6°F), pulse is 115/min, and blood pressure is 140/70 mm Hg. Breath sounds are normal. The remainder of the examination shows no abnormalities. An ECG shows left ventricular hypertrophy with ST-segment elevation in leads I, aVL, and V1–V6. High-dose aspirin, clopidogrel, metoprolol, sublingual nitroglycerin, and unfractionated heparin are administered. As the patient awaits transport to the nearest emergency room, he collapses and becomes unresponsive. His pulse and blood pressure cannot be detected. Despite resuscitative efforts, the patient dies. Which of the following is the most likely cause of death in this patient?
A. Papillary muscle rupture
B. Ventricular fibrillation
C. Septal wall rupture
D. Pulmonary embolism"""


### 3.1 Real-time inference via Amazon SageMaker Endpoint

In [6]:
def print_colored(text, color='green'):
    colors = {
        'green': '\033[92m',
        'reset': '\033[0m',
    }
    color_code = colors.get(color, colors['reset'])
    print(f"{color_code}{text}{colors['reset']}", end="", flush=True)


def invoke_realtime_endpoint(record):
    response = sm_runtime.invoke_endpoint(
        EndpointName=model_name,
        ContentType="application/json",
        Accept="application/json",
        Body=json.dumps(record),
    )
    return json.load(response["Body"])


#### Example 1


In [7]:
input_data = {
    "messages": [
        {"role": "user", "content": prompt1},
    ],
    "stream": False,
}

result    = invoke_realtime_endpoint(input_data)
message   = result["choices"][0]["message"]
reasoning = message.get("reasoning")
content   = message.get("content")
print_colored(reasoning, "green")
print_colored(content, "reset")


Here's a thinking process:

1.  **Analyze User Input:**
   - **Patient:** 95-year-old woman, long-term care resident.
   - **Event:** Fell on right knee, severe right hip/buttock pain, cannot stand.
   - **Signs/Symptoms:** Foreshortened right leg fixed in adduction, large swelling in right buttock. Confused (knows name, doesn't know date, insists to leave).
   - **PMH:** Diabetes, CHF, incontinence.
   - **Meds:** Metformin, lisinopril, hydrochlorothiazide, metoprolol, oxybutynin.
   - **Imaging:** Right posterior hip dislocation without fractures.
   - **Question:** What medication is most likely associated with this patient’s confusion?
   - **Options:** A. Metformin, B. Oxybutynin, C. Metoprolol, D. Lisinopril.

2.  **Identify Key Clinical Concepts:**
   - Elderly patient (95yo)
   - Acute confusion/delirium
   - Medication list: metformin (DM), lisinopril (ACEi), HCTZ (diuretic), metoprolol (beta-blocker), oxybutynin (antimuscarinic for incontinence).
   - Anticholinergic drugs ar

#### Example 2


In [9]:
input_data = {
    "messages": [
        {"role": "user", "content": prompt2},
    ],
    "stream": False,
}

result    = invoke_realtime_endpoint(input_data)
message   = result["choices"][0]["message"]
reasoning = message.get("reasoning")
content   = message.get("content")
print_colored(reasoning, "green")
print_colored(content, "reset")


Here's a thinking process:

1.  **Analyze User Input:**
   - **Patient:** 47-year-old man
   - **Symptoms:** Severe retrosternal chest pain, SOB for 45 mins
   - **PMH/Risk Factors:** Dyslipidemia, HTN, T2DM, 20 pack-year smoking history
   - **Meds:** HCTZ, lisinopril, metformin, atorvastatin
   - **Vitals/Exam:** Pale, diaphoretic, T 37°C, HR 115, BP 140/70. Normal breath sounds. No other abnormalities.
   - **ECG:** LVH, ST-elevation in I, aVL, V1-V6 (suggests extensive anterior/lateral MI)
   - **Treatment given:** Aspirin, clopidogrel, metoprolol, SL NTG, UFH
   - **Event:** Collapses, becomes unresponsive, pulseless/hypotensive (cardiac arrest) while awaiting transport. Dies despite resuscitation.
   - **Question:** Most likely cause of death?
   - **Options:** A. Papillary muscle rupture, B. Ventricular fibrillation, C. Septal wall rupture, D. Pulmonary embolism

2.  **Identify Key Clinical Features & Timeline:**
   - Acute ST-elevation myocardial infarction (STEMI) with extensi

### 3.2 Streaming response

Set `"stream": True` to receive the response as Server-Sent Events (SSE).

In [11]:
def invoke_streaming_endpoint(record):
    try:
        response = sm_runtime.invoke_endpoint_with_response_stream(
            EndpointName=model_name,
            Body=json.dumps(record),
            ContentType="application/json",
            Accept="text/event-stream",
        )
        buffer = ""
        for event in response["Body"]:
            if "PayloadPart" in event:
                raw = event["PayloadPart"]["Bytes"].decode("utf-8")
                buffer += raw
                while "\n" in buffer:
                    line, buffer = buffer.split("\n", 1)
                    line = line.strip()
                    if not line or line == "data: [DONE]":
                        continue
                    if line.startswith("data: "):
                        try:
                            yield json.loads(line[6:])
                        except json.JSONDecodeError:
                            pass
    except Exception as e:
        print(f"Streaming error: {e}")


def handle_streaming_chunk(chunk):
    choices = chunk.get("choices", [])
    if choices:
        delta = choices[0].get("delta", {})
        if delta.get("reasoning"):
            print_colored(delta["reasoning"], "green")
        elif delta.get("content"):
            print_colored(delta["content"], "reset")


In [12]:
payload = {
    "messages": [
        {"role": "user", "content": prompt1},
    ],
    "stream": True,
}

for chunk in invoke_streaming_endpoint(payload):
    handle_streaming_chunk(chunk)


Here's a thinking process:

1.  **Analyze User Input:**
   - **Patient:** 95-year-old woman, long-term care resident.
   - **Event:** Tripped on a rug, fell on right knee. Severe right hip/buttock pain. Cannot stand. Left hip drops when trying to stand on right leg. Right leg foreshortened, fixed in adducted position, large buttock swelling.
   - **Diagnosis:** Right posterior hip dislocation (no fracture). Confirmed by radiograph.
   - **Symptoms/Signs:** Confused, knows name but not date, insists on leaving.
   - **PMH:** Diabetes, CHF, incontinence.
   - **Medications:** Metformin, lisinopril, hydrochlorothiazide, metoprolol, oxybutynin.
   - **Question:** What medication is most likely associated with this patient's confusion?
   - **Options:** A. Metformin, B. Oxybutynin, C. Metoprolol, D. Lisinopril

2.  **Identify Key Clinical Concepts:**
   - Elderly patient (95yo) with acute confusion (delirium).
   - Medication list includes several drugs with CNS effects.
   - Oxybutynin is 

Now that you have successfully performed a real-time inference, you do not need the endpoint any more. You can terminate the endpoint to avoid being charged.

In [27]:
model.sagemaker_session.delete_endpoint(model_name)
model.sagemaker_session.delete_endpoint_config(model_name)

## 4. Batch inference

Batch transform runs inference over files uploaded to S3. Each input file contains one JSON request (same format as real-time). Results are written to the output S3 path with a `.out` suffix.

In [7]:
validation_json_file_name1 = "input1.json"
validation_json_file_name2 = "input2.json"
validation_input_json_path  = f"s3://{s3_bucket}/{model_name}/validation-input/"
validation_output_json_path = f"s3://{s3_bucket}/{model_name}/validation-output/"


def write_and_upload_to_s3(input_data, file_name):
    s3_client.put_object(
        Bucket=s3_bucket,
        Key=f"{model_name}/validation-input/{file_name}",
        Body=bytes(input_data.encode("UTF-8")),
    )


In [8]:
payload1 = json.dumps({
    "messages": [
        {"role": "user", "content": prompt1},
    ],
    "stream": False,
})

payload2 = json.dumps({
    "messages": [
        {"role": "user", "content": prompt2},
    ],
    "stream": False,
})

write_and_upload_to_s3(payload1, validation_json_file_name1)
write_and_upload_to_s3(payload2, validation_json_file_name2)

In [ ]:
transformer = model.transformer(
    instance_count=1,
    instance_type=batch_transform_inference_instance_type,
    accept="application/json",
    output_path=validation_output_json_path,
)
transformer.transform(validation_input_json_path, content_type="application/json")
transformer.wait()

In [ ]:
from urllib.parse import urlparse


def retrieve_json_output_from_s3(validation_file_name):
    parsed_url = urlparse(transformer.output_path)
    file_key   = f"{parsed_url.path[1:]}{validation_file_name}.out"
    response   = s3_client.get_object(Bucket=s3_bucket, Key=file_key)
    result     = json.loads(response["Body"].read().decode("utf-8"))

    for idx, choice in enumerate(result.get("choices", [])):
        print(f"Response {idx + 1}:")
        message   = choice.get("message", {})
        reasoning = message.get("reasoning")
        content   = message.get("content")
        print_colored(reasoning, "green")
        print_colored(content, "reset")
        print()


In [15]:
retrieve_json_output_from_s3(validation_json_file_name1)

Response 1:
Here's a thinking process:

1.  **Analyze User Input:**
   - **Patient:** 95-year-old female in a long-term care facility.
   - **Event:** Tripped on a rug, fell on right knee, severe right hip/buttock pain.
   - **Physical Findings:** Foreshortened right leg fixed in adduction, large swelling in right buttock. Confusion (disoriented to time, agitated/wants to leave).
   - **Imaging:** Right posterior hip dislocation without fractures.
   - **PMH:** Diabetes, CHF, incontinence.
   - **Meds:** Metformin, lisinopril, hydrochlorothiazide, metoprolol, oxybutynin.
   - **Question:** What medication is most likely associated with this patient's confusion?
   - **Options:** A. Metformin, B. Oxybutynin, C. Metoprolol, D. Lisinopril.

2.  **Identify Key Clinical Concepts:**
   - The patient is elderly (95 yo).
   - She presents with acute confusion/delirium.
   - Medication list includes oxybutynin.
   - Oxybutynin is an antimuscarinic (anticholinergic) drug used for overactive blad

In [16]:
retrieve_json_output_from_s3(validation_json_file_name2)

Response 1:
Here's a thinking process:

1.  **Analyze User Input:**
   - **Case:** 47yo man with severe retrosternal chest pain, SOB for 45 mins. Risk factors: dyslipidemia, HTN, T2DM, smoker. Vitals: pale, diaphoretic, HR 115, BP 140/70. ECG: LVH with ST elevation in I, aVL, V1-V6 (extensive anterior/lateral STEMI). Meds given: ASA, clopidogrel, metoprolol, NTG, heparin.
   - **Event:** Collapses, becomes unresponsive, pulseless, BP undetectable while awaiting transport. Dies despite resuscitation.
   - **Question:** Most likely cause of death?
   - **Options:** A. Papillary muscle rupture, B. Ventricular fibrillation, C. Septal wall rupture, D. Pulmonary embolism.

2.  **Identify Key Clinical Features:**
   - Acute STEMI (anterior/lateral)
   - Sudden collapse within ~45 minutes of symptom onset
   - Pulselessness/unresponsiveness -> cardiac arrest
   - Rapid progression to death despite resuscitation

3.  **Evaluate Options in Context of Acute MI Complications & Timing:**
   - *Papi

Congratulations! You just verified that the batch transform job is working as expected. Since the model is not required, you can delete it. Note that you are deleting the deployable model. Not the model package.

In [ ]:
model.delete_model()

### Unsubscribe to the listing (optional)

If you would like to unsubscribe to the model package, follow these steps. Before you cancel the subscription, ensure that you do not have any [deployable model](https://console.aws.amazon.com/sagemaker/home#/models) created from the model package or using the algorithm.

**Steps to unsubscribe to product from AWS Marketplace**:
1. Navigate to __Machine Learning__ tab on [__Your Software subscriptions page__](https://aws.amazon.com/marketplace/ai/library?productType=ml&ref_=mlmp_gitdemo_indust)
2. Locate the listing that you want to cancel the subscription for, and then choose __Cancel Subscription__  to cancel the subscription.